<a href="https://colab.research.google.com/github/Madhav-Sharma91/RFL-python-internship/blob/main/day29project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import re

# =========================================================
# PAGE CONFIG
# =========================================================

st.set_page_config(
    page_title="Smart Expense Tracker",
    page_icon="💰",
    layout="wide"
)

st.title("💰 Smart Expense Tracker & Budget Analyzer")
st.caption(
    "Track expenses, automatically categorize spending, analyze budgets, "
    "visualize trends, and predict future expenses."
)


# =========================================================
# AUTOMATIC EXPENSE CATEGORIZATION
# =========================================================

CATEGORY_KEYWORDS = {

    "Food": [
        "food", "restaurant", "swiggy", "zomato", "dominos",
        "pizza", "burger", "cafe", "coffee", "grocery",
        "groceries", "supermarket", "blinkit", "zepto"
    ],

    "Rent": [
        "rent", "house rent", "apartment", "housing"
    ],

    "Transportation": [
        "uber", "ola", "taxi", "cab", "metro", "bus",
        "train", "petrol", "fuel", "diesel", "parking"
    ],

    "Utilities": [
        "electricity", "water", "gas", "internet",
        "wifi", "mobile", "phone", "recharge", "utility"
    ],

    "Shopping": [
        "amazon", "flipkart", "myntra", "shopping",
        "clothes", "clothing", "shoes", "electronics"
    ],

    "Entertainment": [
        "netflix", "spotify", "movie", "cinema", "prime",
        "youtube", "game", "gaming", "concert"
    ],

    "Healthcare": [
        "hospital", "doctor", "medicine", "medical",
        "pharmacy", "healthcare", "clinic"
    ],

    "Education": [
        "school", "college", "course", "udemy",
        "coursera", "books", "education", "tuition"
    ],

    "Salary/Income": [
        "salary", "income", "bonus", "freelance",
        "payment received", "interest"
    ]
}


def categorize_expense(description):

    text = str(description).lower()

    for category, keywords in CATEGORY_KEYWORDS.items():

        for keyword in keywords:

            if re.search(
                r"\b" + re.escape(keyword) + r"\b",
                text
            ):
                return category

    return "Other"


# =========================================================
# DEMO DATA
# =========================================================

def create_demo_data():

    data = [
        ["2026-01-01", "Salary", 50000],
        ["2026-01-02", "House Rent", 15000],
        ["2026-01-03", "Swiggy food order", 550],
        ["2026-01-04", "Uber ride", 300],
        ["2026-01-05", "Electricity bill", 1800],
        ["2026-01-07", "Amazon shopping", 2500],
        ["2026-01-10", "Grocery store", 3500],
        ["2026-01-12", "Netflix", 649],
        ["2026-01-15", "Petrol", 2000],
        ["2026-01-18", "Restaurant dinner", 1200],
        ["2026-01-20", "Pharmacy", 850],
        ["2026-01-22", "Online course", 1500],

        ["2026-02-01", "Salary", 50000],
        ["2026-02-02", "House Rent", 15000],
        ["2026-02-03", "Zomato", 700],
        ["2026-02-05", "Electricity", 1700],
        ["2026-02-07", "Amazon shopping", 1800],
        ["2026-02-10", "Groceries", 4000],
        ["2026-02-12", "Uber", 450],
        ["2026-02-15", "Petrol", 2200],
        ["2026-02-20", "Cinema", 900],

        ["2026-03-01", "Salary", 50000],
        ["2026-03-02", "Rent", 15000],
        ["2026-03-04", "Swiggy", 650],
        ["2026-03-06", "Electricity bill", 1900],
        ["2026-03-08", "Myntra shopping", 3200],
        ["2026-03-11", "Grocery", 3700],
        ["2026-03-15", "Petrol", 2400],
        ["2026-03-18", "Netflix", 649],
        ["2026-03-22", "Doctor consultation", 1200],
    ]

    return pd.DataFrame(
        data,
        columns=["Date", "Description", "Amount"]
    )


# =========================================================
# SIDEBAR
# =========================================================

st.sidebar.header("⚙️ Controls")

uploaded_file = st.sidebar.file_uploader(
    "Upload Expense CSV",
    type=["csv"]
)

if uploaded_file is not None:

    df = pd.read_csv(uploaded_file)

else:

    st.info(
        "No CSV uploaded. Demo data is being displayed. "
        "Upload your own CSV from the sidebar to analyze your expenses."
    )

    df = create_demo_data()


# =========================================================
# COLUMN MAPPING
# =========================================================

st.sidebar.subheader("Column Mapping")

columns = df.columns.tolist()

# Helper function to get the index of a column safely
def get_col_index(col_name, col_list):
    try:
        return col_list.index(col_name)
    except ValueError:
        return 0 # Default to the first column if not found

date_column = st.sidebar.selectbox(
    "Date column",
    columns,
    index=get_col_index("Date", columns)
)

description_column = st.sidebar.selectbox(
    "Description column",
    columns,
    index=get_col_index("Description", columns)
)

amount_column = st.sidebar.selectbox(
    "Amount column",
    columns,
    index=get_col_index("Amount", columns)
)


df = df.rename(
    columns={
        date_column: "Date",
        description_column: "Description",
        amount_column: "Amount"
    }
)

df["Date"] = pd.to_datetime(
    df["Date"],
    errors="coerce"
)

df["Amount"] = pd.to_numeric(
    df["Amount"],
    errors="coerce"
)

df = df.dropna(
    subset=["Date", "Description", "Amount"]
)

df["Category"] = df["Description"].apply(
    categorize_expense
)

df["Month"] = df["Date"].dt.to_period(
    "M"
).astype(str)


# =========================================================
# INCOME / EXPENSE IDENTIFICATION
# =========================================================

income_categories = [
    "Salary/Income"
]

df["Type"] = np.where(
    df["Category"].isin(income_categories),
    "Income",
    "Expense"
)


# =========================================================
# BUDGET INPUT
# =========================================================

st.sidebar.subheader("💵 Budget Settings")

monthly_budget = st.sidebar.number_input(
    "Monthly Expense Budget",
    min_value=0.0,
    value=30000.0,
    step=1000.0
)


# =========================================================
# DATA PREVIEW
# =========================================================

st.subheader("📄 Imported Expense Data")

st.dataframe(
    df,
    use_container_width=True,
    hide_index=True
)


# =========================================================
# MONTHLY SUMMARY
# =========================================================

monthly_income = (
    df[df["Type"] == "Income"]
    .groupby("Month")["Amount"]
    .sum()
)

monthly_expenses = (
    df[df["Type"] == "Expense"]
    .groupby("Month")["Amount"]
    .sum()
)

monthly_summary = pd.DataFrame({
    "Income": monthly_income,
    "Expenses": monthly_expenses
}).fillna(0)

monthly_summary["Savings"] = (
    monthly_summary["Income"]
    - monthly_summary["Expenses"]
)

monthly_summary["Savings_%"] = np.where(
    monthly_summary["Income"] > 0,
    monthly_summary["Savings"]
    / monthly_summary["Income"] * 100,
    0
)

monthly_summary["Budget"] = monthly_budget

monthly_summary["Budget_Status"] = np.where(
    monthly_summary["Expenses"] <= monthly_budget,
    "Within Budget",
    "Over Budget"
)


# =========================================================
# TOP KPIs
# =========================================================

st.subheader("📊 Financial Overview")

total_income = df.loc[
    df["Type"] == "Income",
    "Amount"
].sum()

total_expenses = df.loc[
    df["Type"] == "Expense",
    "Amount"
].sum()

total_savings = total_income - total_expenses

savings_rate = (
    total_savings / total_income * 100
    if total_income > 0
    else 0
)

col1, col2, col3, col4 = st.columns(4)

col1.metric(
    "Total Income",
    f"₹{total_income:,.2f}"
)

col2.metric(
    "Total Expenses",
    f"₹{total_expenses:,.2f}"
)

col3.metric(
    "Total Savings",
    f"₹{total_savings:,.2f}"
)

col4.metric(
    "Savings Rate",
    f"{savings_rate:.2f}%"
)


# =========================================================
# BUDGET ANALYSIS
# =========================================================

st.subheader("💰 Budget Summary")

latest_month = monthly_summary.index[-1]

latest_expenses = monthly_summary.loc[
    latest_month,
    "Expenses"
]

budget_remaining = monthly_budget - latest_expenses

if budget_remaining >= 0:

    st.success(
        f"✅ {latest_month}: ₹{budget_remaining:,.2f} "
        f"remaining from your monthly budget."
    )

else:

    st.error(
        f"⚠️ {latest_month}: You exceeded your budget by "
        f"₹{abs(budget_remaining):,.2f}."
    )


# =========================================================
# MONTHLY SUMMARY TABLE
# =========================================================

summary_display = monthly_summary.reset_index()

summary_display.columns = [
    "Month",
    "Income",
    "Expenses",
    "Savings",
    "Savings %",
    "Budget",
    "Budget Status"
]

st.dataframe(
    summary_display.style.format({
        "Income": "₹{:,.2f}",
        "Expenses": "₹{:,.2f}",
        "Savings": "₹{:,.2f}",
        "Savings %": "{:.2f}%",
        "Budget": "₹{:,.2f}"
    }),
    use_container_width=True,
    hide_index=True
)


# =========================================================
# SPENDING BY CATEGORY
# =========================================================

st.subheader("🥧 Spending by Category")

category_data = (
    df[df["Type"] == "Expense"]
    .groupby("Category")["Amount"]
    .sum()
    .reset_index()
    .sort_values("Amount", ascending=False)
)

fig_category = px.pie(
    category_data,
    names="Category",
    values="Amount",
    hole=0.4,
    title="Expense Distribution by Category"
)

st.plotly_chart(
    fig_category,
    use_container_width=True
)


# =========================================================
# MONTHLY SPENDING TREND
# =========================================================

st.subheader("📈 Monthly Spending Trends")

monthly_chart = (
    df[df["Type"] == "Expense"]
    .groupby("Month")["Amount"]
    .sum()
    .reset_index()
)

fig_monthly = px.line(
    monthly_chart,
    x="Month",
    y="Amount",
    markers=True,
    title="Monthly Expense Trend",
    labels={
        "Amount": "Expenses (₹)",
        "Month": "Month"
    }
)

fig_monthly.add_hline(
    y=monthly_budget,
    line_dash="dash",
    line_color="red",
    annotation_text="Monthly Budget"
)

st.plotly_chart(
    fig_monthly,
    use_container_width=True
)


# =========================================================
# CATEGORY VS BUDGET
# =========================================================

st.subheader("📊 Category-wise Spending")

category_month = (
    df[df["Type"] == "Expense"]
    .groupby(["Month", "Category"])["Amount"]
    .sum()
    .reset_index()
)

fig_category_bar = px.bar(
    category_month,
    x="Month",
    y="Amount",
    color="Category",
    barmode="stack",
    title="Monthly Spending by Category"
)

st.plotly_chart(
    fig_category_bar,
    use_container_width=True
)


# =========================================================
# DAILY EXPENSE TREND
# =========================================================

st.subheader("📅 Daily Expense Trend")

daily_expenses = (
    df[df["Type"] == "Expense"]
    .groupby("Date")["Amount"]
    .sum()
    .reset_index()
)

fig_daily = px.line(
    daily_expenses,
    x="Date",
    y="Amount",
    markers=True,
    title="Daily Spending"
)

st.plotly_chart(
    fig_daily,
    use_container_width=True
)


# =========================================================
# TOP EXPENSES
# =========================================================

st.subheader("🔎 Top Expenses")

top_expenses = (
    df[df["Type"] == "Expense"]
    .sort_values("Amount", ascending=False)
    .head(10)
)

st.dataframe(
    top_expenses[
        [
            "Date",
            "Description",
            "Category",
            "Amount"
        ]
    ].style.format({
        "Amount": "₹{:,.2f}"
    }),
    use_container_width=True,
    hide_index=True
)


# =========================================================
# EXPENSE PREDICTION
# =========================================================

st.subheader("🔮 Expense Prediction")

expense_history = (
    df[df["Type"] == "Expense"]
    .groupby("Month")["Amount"]
    .sum()
    .reset_index()
)

expense_history["Month_Number"] = np.arange(
    len(expense_history)
)

if len(expense_history) >= 2:

    x = expense_history["Month_Number"].values
    y = expense_history["Amount"].values

    slope, intercept = np.polyfit(
        x,
        y,
        1
    )

    next_month_number = len(expense_history)

    predicted_expense = (
        slope * next_month_number
        + intercept
    )

    predicted_expense = max(
        0,
        predicted_expense
    )

    col1, col2 = st.columns(2)

    col1.metric(
        "Predicted Next-Month Expense",
        f"₹{predicted_expense:,.2f}"
    )

    difference = (
        predicted_expense - monthly_budget
    )

    if difference > 0:

        col2.error(
            f"⚠️ Predicted spending is "
            f"₹{difference:,.2f} above your budget."
        )

    else:

        col2.success(
            f"✅ Predicted spending is "
            f"₹{abs(difference):,.2f} below your budget."
        )

    prediction_data = expense_history.copy()

    prediction_data["Predicted"] = (
        slope * prediction_data["Month_Number"]
        + intercept
    )

    fig_prediction = px.line(
        prediction_data,
        x="Month",
        y=["Amount", "Predicted"],
        markers=True,
        title="Historical vs Predicted Expenses"
    )

    st.plotly_chart(
        fig_prediction,
        use_container_width=True
    )

else:

    st.warning(
        "At least two months of expense data are required "
        "for expense prediction."
    )


# =========================================================
# SMART INSIGHTS
# =========================================================

st.subheader("💡 Smart Spending Insights")

if not category_data.empty:

    highest_category = category_data.iloc[0]

    st.info(
        f"Your highest spending category is "
        f"**{highest_category['Category']}**, accounting for "
        f"₹{highest_category['Amount']:,.2f}."
    )


if total_income > 0:

    if savings_rate >= 30:

        st.success(
            "🎉 Excellent! You are saving at least 30% "
            "of your income."
        )

    elif savings_rate >= 10:

        st.info(
            "👍 Your savings rate is positive. "
            "Consider gradually increasing it."
        )

    else:

        st.warning(
            "⚠️ Your savings rate is low. "
            "Consider reviewing discretionary expenses."
        )


# =========================================================
# EXPORT FINAL REPORT
# =========================================================

st.subheader("⬇️ Export Final Report")

report = df.copy()

report_csv = report.to_csv(
    index=False
)

summary_csv = summary_display.to_csv(
    index=False
)

col1, col2 = st.columns(2)

with col1:

    st.download_button(
        label="📥 Download Detailed Expense Report",
        data=report_csv,
        file_name="expense_report.csv",
        mime="text/csv"
    )

with col2:

    st.download_button(
        label="📥 Download Monthly Budget Summary",
        data=summary_csv,
        file_name="monthly_budget_summary.csv",
        mime="text/csv"
    )


# =========================================================
# FOOTER
# =========================================================

st.divider()

st.caption(
    "Smart Expense Tracker • Automatic categorization • "
    "Budget analysis • Trend visualization • Expense prediction"
)


2026-09-04 05:59:25.998 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-04 05:59:25.999 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-04 05:59:25.999 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-04 05:59:26.001 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-04 05:59:26.002 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-04 05:59:26.002 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-04 05:59:26.003 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-04 05:59:26.005 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

DeltaGenerator()